## Gentle Introduction to torch.autograd

In [2]:
import torch
from torchvision.models import resnet18, ResNet18_Weights

### Autograd Explained

In [24]:
# Lets define two tensors
a = torch.tensor(2.0)
b = torch.tensor(3.0)
# and multipy them together
c = a*b

# some behaviors to make note of - any computation that involves a tensor that requires_grad
# will result in an output tensor that also has that enabled
print(c.requires_grad)
print(c.is_leaf)
print(c.grad_fn)
d = torch.tensor(1.0, requires_grad=True)
c = a*d
print(c.requires_grad)
# it will now have the grad_fn ( akin to our _backwards from the micrograd expirement)
# and it will now no longer consider it self a leaf 
print(c.is_leaf)
print(c.grad_fn)

## This multiplication builds a backwards graph ( topologically sorted DAG ) in the background
## the operators have a context object that be used as a store for anything it needs during its
## backpropogation -> this context object gets passed to the MulBackward object - that 
## stored in the grad_fn attr
# ctx has these attrs ctx.save_for_backward method and ctx.saved_tensors -> these are the python version of how the input 
# input tensors are referenced in the backward pass - theyre a symbolic representation

# the MulBackward is an object attrs
'next_functions -> contains list of tuples that are associated to the inputs passed in'
' the second element of the tuple relates to the output index if we unbind a tensor using torch.unbind'
'register_hook -> this and the pre hook will be learned later'
'register_prehook-'
'requires_grad'
print(c.grad_fn.next_functions)
# as you can see node d's func is accume grad - and a that doesn't need a grad has none
# note accume grad is just add grad values



False
True
None
True
False
((None, 0), (<AccumulateGrad object at 0x1662ba620>, 0))


In [26]:
## it also has a version number attr that updates whenever we do an inplace operation
## if we try to call backward on it and it detects a version inconsistency is should throw an 
## error
d +=1
c.backward()
# but note that this wouldn't be an issue if c was defined by addition 
# as the addition operator because our backwards graph doesn't depend on knowing the value
# of that c tensor as the gradient is just pass through 

RuntimeError: a leaf Variable that requires grad is being used in an in-place operation.

In [3]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)
data = torch.rand(1,3,64,64)
labels = torch.rand(1,1000)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/diz0/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████████████████████████████████| 44.7M/44.7M [00:18<00:00, 2.53MB/s]


In [4]:
# forward pass
prediction = model(data)

In [11]:
# loss
loss = (prediction-labels).sum()
loss

tensor(-499.7525, grad_fn=<SumBackward0>)

In [6]:
# backprop
loss.backward()

tensor(-499.7525, grad_fn=<SumBackward0>)

In [8]:
# load the optimizer - in this case SGD
optim = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)
optim.step() # gradient descent

In [9]:
## Note DAGs are dynamic in PyTorch - meaning after each backward() call, autograd starts populating a new graph -
## Which is the same thing we do when de zero out the grads

In [ ]:
## Exclusion from gradient calculations -> torch keeps track of which tensors need to keep their operation history
## this is done by setting the requires_grad kwarg when initiating an tensor to True

# Tensors who don't need to keep track of this/ require grad - are called Frozen parameters

## Training the resnet18 model we imported earlier

In [ ]:
# When fine tuning - we usually freeze most of the model - and typically only modify
# the classifier layers to make predictions on the new labels

# Freeeeze
for param in model.parameters():
    param.requires_grad = False

# in resnet - the classifier is the last layer in the net - identified by model.fc
# lets now replace it with a new layer
model.fc = torch.nn.Linear(512,10)